In [ ]:
# Instalar el SDK (si no está instalado)
!pip install typesafe-sdk


In [ ]:
# Importaciones principales
from typesafe_sdk import (
    TypeSafeClient,
    AsyncTypeSafeClient,
    Noul,
    Choice,
    Score,
    Question,
    RetryPolicy,
    TypeSafeError,
    TypeSafeAuthenticationError,
    TypeSafeAPIError,
)
import os
import asyncio

print("✅ Importaciones completadas exitosamente")

## Configurar el cliente


In [ ]:
# Agrega tu API Key como secreto en Google Colab para que no quede expuesta
from google.colab import userdata
import os

os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")

In [ ]:
# La clave ya está en el entorno (secreto de Colab)
API_KEY = os.getenv("TYPESAFE_API_KEY")

client = TypeSafeClient(api_key=API_KEY)

print("✅ Cliente TypeSafe creado exitosamente")


## Noul: probabilidad de un sí o un no




In [ ]:
# Ejemplo: Analizar urgencia en tickets de soporte
ticket_urgente = """
¡NECESITO AYUDA INMEDIATA! Mi sistema está caído desde hace 3 horas
y estamos perdiendo miles de dólares por minuto. Esto es inaceptable.
Por favor escalen esto AHORA MISMO.
"""

ticket_normal = """
Hola, tengo una pregunta sobre cómo usar la función de exportación.
Cuando pueda, me gustaría recibir información al respecto.
Gracias.
"""

# Definir pregunta tipo Noul
pregunta_urgencia = Noul(
    instructions="¿Este ticket exige atención inmediata?"
)

print("📋 Pregunta Noul definida:")
print(f"   Instrucciones: {pregunta_urgencia.instructions}")
print(f"   Tipo: {type(pregunta_urgencia).__name__}")


In [ ]:
with TypeSafeClient(api_key=API_KEY) as client:
    resultado = client.system_one(
        state=ticket_urgente,
        questions={"urgencia": pregunta_urgencia}
    )

    print(f"Nivel de urgencia: {resultado.nouls['urgencia'].noul:.2%}")
    # Esperado: ~0.95 (95% urgente)

print("💡 Ejemplo de uso de Noul mostrado arriba")


## Choice: una categoría entre varias


In [ ]:
# Ejemplo: Clasificar tono de comentarios de clientes
comentarios = [
    "¡Excelente servicio! Muy satisfecho con la atención.",
    "El producto llegó dañado y nadie me ayuda. Pésimo servicio.",
    "Recibí el pedido, todo bien."
]

# Definir pregunta tipo Choice con criterios
pregunta_tono = Choice(
    instructions="¿Cuál es el tono emocional de este comentario?",
    criteria={
        "positivo": "El cliente expresa satisfacción, alegría o gratitud",
        "negativo": "El cliente expresa enojo, frustración o decepción",
        "neutral": "El cliente informa hechos sin emoción aparente"
    }
)

print("📋 Pregunta Choice definida:")
print(f"   Instrucciones: {pregunta_tono.instructions}")
print(f"   Criterios: {list(pregunta_tono.criteria.keys())}")


In [ ]:
with TypeSafeClient(api_key=API_KEY) as client:
    for i, comentario in enumerate(comentarios, 1):
        resultado = client.system_one(
            state=comentario,
            questions={"tono": pregunta_tono}
        )
        tono_detectado = resultado.choices["tono"].choice
        print(f"Comentario {i}: '{comentario[:50]}...'")
        print(f"   → Tono detectado: {tono_detectado}\n")

print("💡 Ejemplo de uso de Choice mostrado arriba")


## Score: una posición en una escala que tú defines


In [ ]:
# Ejemplo: Evaluar calidad de leads de ventas
lead_alta_calidad = """
Soy el director de TI de una empresa Fortune 500.
Estamos buscando implementar su solución enterprise antes de Q4.
Tenemos presupuesto aprobado de $500k y necesitamos una demo esta semana.
Por favor contacten a mi equipo para coordinar.
"""

lead_baja_calidad = """
Hola, soy estudiante y estoy investigando para mi tesis.
¿Podrían enviarme información sobre sus productos?
Quizás en el futuro cuando tenga empresa los considere.
"""

# Definir pregunta tipo Score
pregunta_calidad = Score(
    instructions="Evalúa la calidad de este lead de ventas",
    criteria=[
        "1-3: Lead poco calificado: sin presupuesto, sin timeline, sin autoridad",
        "4-6: Lead medio: algún interés pero falta información clave",
        "7-9: Lead bueno: tiene presupuesto y timeline definidos",
        "10: Lead excelente: presupuesto alto, timeline urgente, decision maker"
    ]
)

print("📋 Pregunta Score definida:")
print(f"   Criterios: {len(pregunta_calidad.criteria)} niveles")
print(f"   Instrucciones: {pregunta_calidad.instructions}")

In [ ]:
with TypeSafeClient(api_key=API_KEY) as client:
    for nombre, lead in [("Alta", lead_alta_calidad), ("Baja", lead_baja_calidad)]:
        resultado = client.system_one(
            state=lead,
            questions={"calidad": pregunta_calidad}
        )
        puntaje = resultado.scores["calidad"].score
        print(f"Lead {nombre}: score = {puntaje:.2f} (escala 0–3)")


## Varias preguntas en una sola llamada


In [ ]:
# Ejemplo: Análisis completo de un email de cliente
email_cliente = """
Estimado equipo de soporte,

Llevo 3 días intentando acceder a mi cuenta y nadie me responde.
Esto es absolutamente inaceptable para una empresa como la vuestra.
Si no resuelven esto hoy, voy a cancelar mi suscripción enterprise
de $10,000 anuales y me cambiaré a la competencia.

Espero una respuesta INMEDIATA.

Carlos Rodríguez
Director de Operaciones, TechCorp Inc.
"""

# Definir múltiples preguntas
analisis_completo = {
    "urgencia": Noul(
        instructions="¿Este caso exige atención inmediata?"
    ),
    "tono": Choice(
        instructions="¿Cuál es el tono del cliente?",
        criteria={
            "calmado": "Cliente tranquilo y razonable",
            "frustrado": "Cliente molesto pero constructivo",
            "enojado": "Cliente furioso y amenazante"
        }
    ),
    "segmento_cliente": Score(
        instructions="¿En qué segmento de valor encaja este cliente?",
        criteria=[
            "Cliente gratuito o trial",
            "Cliente SMB",
            "Cliente mid-market",
            "Cliente enterprise",
        ]
    ),
    "riesgo_churn": Noul(
        instructions="¿Hay una amenaza clara de que este cliente cancele?"
    )
}

print("📋 Análisis múltiple definido:")
for nombre, pregunta in analisis_completo.items():
    tipo = type(pregunta).__name__
    print(f"   • {nombre}: {tipo}")


In [ ]:
with TypeSafeClient(api_key=API_KEY) as client:
    resultado = client.system_one(
        state=email_cliente,
        questions=analisis_completo
    )

    print("\n📊 RESULTADOS DEL ANÁLISIS:")
    print(f"   Urgencia: {resultado.nouls['urgencia'].noul:.2%}")
    print(f"   Tono: {resultado.choices['tono'].choice}")
    print(f"   Segmento (score): {resultado.scores['segmento_cliente'].score}")
    print(f"   Riesgo de churn: {resultado.nouls['riesgo_churn'].noul:.2%}")

    # Recomendación automática
    if resultado.nouls['riesgo_churn'].noul > 0.7:
        print("\n⚠️  RECOMENDACIÓN: Escalar inmediatamente a gerente de cuenta")


## Cliente asíncrono


In [ ]:
async def ejemplo_asincrono():
    """Ejemplo de uso asíncrono del cliente TypeSafe"""

    async with AsyncTypeSafeClient(api_key=API_KEY) as client:
        # Preparar múltiples análisis en paralelo
        textos = [
            "Excelente producto, muy recomendado",
            "Terrible experiencia, no lo compren",
            "Producto promedio, cumple su función"
        ]

        pregunta_sentimiento = Choice(
            instructions="Clasifica el sentimiento",
            criteria={"positivo": None, "negativo": None, "neutral": None}
        )

        # Ejecutar análisis en paralelo
        tareas = [
            client.system_one(state=texto, questions={"sentimiento": pregunta_sentimiento})
            for texto in textos
        ]

        resultados = await asyncio.gather(*tareas)

        for i, (texto, resultado) in enumerate(zip(textos, resultados), 1):
            sentimiento = resultado.choices["sentimiento"].choice
            print(f"{i}. '{texto}' → {sentimiento}")


await ejemplo_asincrono()

## Reintentos y timeouts


In [ ]:
retry_policy = RetryPolicy(max_retries=3)

client_con_retry = TypeSafeClient(
    api_key=API_KEY,
    retry=retry_policy,
    timeout=30.0,
)

print("✅ Cliente con política de retry configurado")
print(f"   Máximos retries: {retry_policy.max_retries}")


## Listar modelos disponibles


In [ ]:
with TypeSafeClient(api_key=API_KEY) as client:
    modelos = client.models.list()

    print("📦 Modelos disponibles:")
    for modelo in modelos.models:
        nombre = getattr(modelo, "name", None) or getattr(modelo, "id", "desconocido")
        descripcion = getattr(modelo, "description", "") or ""
        print(f"   • {nombre}")
        if descripcion:
            print(f"      {descripcion}")


## Errores tipados


In [ ]:
from typesafe_sdk import (
    TypeSafeAuthenticationError,
    TypeSafeAPIError,
    TypeSafeAPIConnectionError,
    TypeSafeAPITimeoutError,
    TypeSafeRateLimitError,
    TypeSafeNotFoundError,
    TypeSafeBadRequestError,
)

def analisis_con_manejo_de_errores(texto: str, api_key: str):
    """Ejemplo robusto de manejo de errores"""
    try:
        with TypeSafeClient(api_key=api_key) as client:
            resultado = client.system_one(
                state=texto,
                questions={
                    "sentimiento": Choice(
                        instructions="Sentimiento",
                        criteria={"positivo": None, "negativo": None, "neutral": None}
                    )
                }
            )
            return resultado.choices["sentimiento"].choice

    except TypeSafeAuthenticationError:
        print("❌ Error: API key inválida o expirada")
        raise
    except TypeSafeRateLimitError as e:
        print(f"⏳ Rate limit excedido. Reintentar en {e.retry_after} segundos")
        raise
    except TypeSafeAPITimeoutError:
        print("⏱️  Timeout: La API no respondió a tiempo")
        raise
    except TypeSafeAPIConnectionError:
        print("🔌 Error de conexión: Verifica tu conexión a internet")
        raise
    except TypeSafeNotFoundError:
        print("🔍 Error: Recurso no encontrado (modelo inválido?)")
        raise
    except TypeSafeBadRequestError as e:
        print(f"📝 Error en la solicitud: {e.message}")
        raise
    except TypeSafeAPIError as e:
        print(f"💥 Error general de API: {e.status_code} - {e.message}")
        raise
    except TypeSafeError as e:
        print(f"⚠️  Error desconocido: {e}")
        raise

print("✅ Función con manejo robusto de errores definida")
print("   Tipos de errores manejados:")
print("   • AuthenticationError")
print("   • RateLimitError")
print("   • APITimeoutError")
print("   • APIConnectionError")
print("   • NotFoundError")
print("   • BadRequestError")
print("   • APIError")
print("   • TypeSafeError (base)")

## Caso de uso: clasificar tickets y decidir qué hacer


In [ ]:
# Definir esquema de análisis para tickets
def analizar_ticket(ticket_texto: str, api_key: str):
    """Analiza un ticket de soporte y devuelve clasificación completa"""

    preguntas_ticket = {
        "categoria": Choice(
            instructions="¿Cuál es la categoría principal del ticket?",
            criteria={
                "tecnico": "Problemas técnicos, bugs, errores de software",
                "facturacion": "Pagos, facturas, reembolsos, cargos",
                "ventas": "Consultas pre-venta, demos, precios",
                "cuenta": "Acceso, contraseñas, configuración de cuenta",
                "general": "Otras consultas generales"
            }
        ),
        "prioridad": Choice(
            instructions="¿Qué prioridad debería tener este ticket?",
            criteria={
                "critica": "Sistema caído, pérdida de datos, múltiples usuarios afectados",
                "alta": "Funcionalidad importante rota, workaround difícil",
                "media": "Problema molesto pero con workaround",
                "baja": "Mejoras, preguntas, cosméticos"
            }
        ),
        "urgencia_num": Noul(
            instructions="¿Este ticket exige atención inmediata?"
        ),
        "sentimiento": Choice(
            instructions="Estado emocional del cliente",
            criteria={
                "muy_enojado": "Furioso, amenazando con irse",
                "molesto": "Frustrado pero razonable",
                "neutral": "Sin emoción aparente",
                "satisfecho": "Contento o agradecido"
            }
        ),
        "requiere_escalamiento": Noul(
            instructions="¿Este ticket requiere escalamiento a soporte nivel 2 o 3?"
        )
    }


    with TypeSafeClient(api_key=api_key) as client:
         resultado = client.system_one(
             state=ticket_texto,
             questions=preguntas_ticket
         )

         return {
             "categoria": resultado.choices["categoria"].choice,
             "prioridad": resultado.choices["prioridad"].choice,
             "urgencia": resultado.nouls["urgencia_num"].noul,
             "sentimiento": resultado.choices["sentimiento"].choice,
             "prob_escalamiento": resultado.nouls["requiere_escalamiento"].noul
         }



In [ ]:
# Ejemplos de tickets para probar
tickets_ejemplo = [
    # Ticket 1: Crítico técnico
    """URGENTE: Nuestra plataforma está completamente caída desde hace 2 horas.
    Todos nuestros 500 usuarios no pueden acceder. Estamos perdiendo $10k por hora.
    Necesitamos ayuda INMEDIATA o cancelaremos el contrato.""",

    # Ticket 2: Consulta de facturación
    """Hola, tengo una pregunta sobre mi última factura. Veo un cargo de $99
    que no reconozco. ¿Podrían explicarme qué es? Gracias.""",

    # Ticket 3: Bug menor
    """Encontré un pequeño bug: cuando hago clic en el botón de exportar
    dos veces rápido, se descarga el archivo dos veces. No es grave pero
    sería bueno arreglarlo.""",

    # Ticket 4: Lead de ventas
    """Somos una empresa de 200 empleados y estamos evaluando su producto
    para implementarlo en Q2. ¿Podrían agendar una demo con nuestro equipo
    técnico? Tenemos presupuesto aprobado."""
]

print(f"📋 {len(tickets_ejemplo)} tickets de ejemplo definidos")

In [ ]:
# Analizar todos los tickets
resultados = []
for i, ticket in enumerate(tickets_ejemplo, 1):
    print(f"\n🎫 TICKET {i}:")
    print(f"   Texto: {ticket[:80]}...")

    resultado = analizar_ticket(ticket, api_key=API_KEY)
    resultados.append(resultado)

    print(f"   Categoría: {resultado['categoria']}")
    print(f"   Prioridad: {resultado['prioridad']}")
    print(f"   Urgencia: {resultado['urgencia']:.2%}")
    print(f"   Sentimiento: {resultado['sentimiento']}")
    print(f"   Prob. escalamiento: {resultado['prob_escalamiento']:.2%}")

    # Reglas automáticas de routing
    if resultado['prioridad'] == 'critica':
        print("   ⚠️  ACCIÓN: Escalar a equipo de guardia inmediatamente")
    elif resultado['prob_escalamiento'] > 0.7:
        print("   ⚠️  ACCIÓN: Asignar a soporte nivel 2")
    elif resultado['categoria'] == 'ventas':
        print("   💼 ACCIÓN: Transferir a equipo de ventas")
    else:
        print("   ✅ ACCIÓN: Asignar a cola estándar")